# ATML PA3 · Task 3 · Step 4: parallel sampling vs. sequential refinement (live generation)

Runs end-to-end with **Run All** / **Save Version** (GPU **T4 x2**, Internet ON).
No dataset needs to be attached: the 50 live questions (and the dev / eval splits) were derived
locally from the supplied ATML-PA3-Data by `task3.prepare_splits` and are committed under `task3/data/`.
The model revision pinned in the supplied README.MD is set in `task3/configs/live.yaml`.

1. clones the repo, installs dependencies, prints the fixed live subset (sorted MATH question IDs
   permuted with `default_rng(6304)`, first 200 = development, the first 50 evaluation questions)
2. `task3.live` on both GPUs (questions split by index, every strategy of a question on the same GPU):
   Qwen2.5-3B-Instruct, FP16, batch size 1, T 0.7 / top-p 0.8 / top-k 20; single response (3,072
   tokens), three parallel responses (1,024 each), refinement of the first parallel response (2,048)
3. `task3.live_eval` → accuracy, input / output tokens, runtime, truncation / extraction rates,
   refinement transitions, bootstrap CIs → **`/kaggle/working/task3_live.zip`**

Every finished call is saved immediately; if a version fails, attach its output and run again: finished
calls are reused.

In [ ]:
# ---- settings ---------------------------------------------------------------------------
REPO_URL   = "https://github.com/alihasnaink/finetune-ground-scale.git"
BRANCH     = "main"
SMOKE_TEST = False   # True = 2 questions with small token caps, separate output folder

REPO_DIR = "/kaggle/working/repo"
OUT_ROOT = "/kaggle/working/task3_smoke" if SMOKE_TEST else "/kaggle/working/task3"
ZIP_PATH = "/kaggle/working/task3_live_smoke.zip" if SMOKE_TEST else "/kaggle/working/task3_live.zip"
OVERRIDES = (
    ["strategies.single_max_new=96", "strategies.parallel_max_new=32", "strategies.refine_max_new=64"]
    if SMOKE_TEST else [])
LIMIT = ["--limit", "2"] if SMOKE_TEST else []

In [ ]:
!nvidia-smi
import os, subprocess, sys, time, glob
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], text=True))

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-task3.txt
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPU = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert N_GPU >= 1, "enable a GPU accelerator (T4 x2 recommended)"

In [ ]:
ENV = os.environ | {"T3_OUT_ROOT": OUT_ROOT, "T3_PRIOR_SEARCH": "/kaggle/input",
                    "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
CFG = "task3/configs/live.yaml"

def cmd(module, *args):
    return [sys.executable, "-m", f"task3.{module}", "--config", CFG, *args] + (["--set", *OVERRIDES] if OVERRIDES else [])

def run(module, *args, gpu="0"):
    p = subprocess.run(cmd(module, *args), cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": gpu})
    if p.returncode != 0:
        raise RuntimeError(f"task3.{module} failed (exit {p.returncode})")

os.makedirs(OUT_ROOT, exist_ok=True)
# the committed live subset (checked against the committed split)
run("live", "--inspect")

In [ ]:
# live generation: one lane per GPU, questions split by index
t0 = time.time()
n_lanes = max(1, min(N_GPU, 2))
lanes = []
for k in range(n_lanes):
    log = open(f"{OUT_ROOT}/live_lane{k}.log", "w")
    lanes.append((k, subprocess.Popen(cmd("live", "--lane", str(k), "--n-lanes", str(n_lanes), *LIMIT),
                                      cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": str(k)},
                                      stdout=log, stderr=subprocess.STDOUT)))
while any(p.poll() is None for _, p in lanes):
    time.sleep(60)
    tails = []
    for k, _ in lanes:
        lines = open(f"{OUT_ROOT}/live_lane{k}.log").read().strip().splitlines()
        tails.append(f"lane {k}: " + (lines[-1][:160] if lines else "starting"))
    print(f"[{(time.time() - t0) / 60:.0f} min]", " || ".join(tails), flush=True)
for k, p in lanes:
    print(open(f"{OUT_ROOT}/live_lane{k}.log").read()[-3000:])
    assert p.returncode == 0, f"lane {k} failed; see {OUT_ROOT}/live_lane{k}.log"
print(f"generation finished in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# scoring (math-verify), costs, transitions, bootstrap CIs; zip the evidence
run("live_eval")
import shutil
shutil.make_archive(ZIP_PATH[:-4], "zip", os.path.dirname(OUT_ROOT), os.path.basename(OUT_ROOT))
from IPython.display import FileLink
display(FileLink(os.path.relpath(ZIP_PATH, "/kaggle/working")))